# Varlen Attention 的训练接入：最新源码走查

最新版本在语言模型侧只提供两种可用的 Attention backend：`flex`（FlexAttention + BlockMask）与 `varlen`（VarlenAttention + 累计长度）。`sdpa` 已不再作为语言模型 backend，`get_attention_config("sdpa")` 会直接抛错。在 Ascend NPU 上，`varlen` 这一支由 `torchtitan_npu.override.qwen3_5.varlen_attention` 接管，把逐文档的累计长度交给 CANN FA v3 的 TND 路径执行。

本节只走查最新源码里的这条路径：样本起点怎样从 `positions` 变成 `VarlenMetadata`，CP 训练怎样把它整理成 `QwenCPMetadata`，`AscVarlenAttention.forward()` 怎样把它翻译成 `npu_fusion_attention_v3` 的参数，最后看训练 recipe 用哪几个开关打开它。

**学习目标**：

- 定位 `attn_backend` 到 inner attention 的映射，以及 varlen 分支的落点；
- 说明 `create_varlen_metadata_for_document()` 从 `positions` 推导 `cu_seqlens` 的规则；
- 读懂 `AscVarlenAttention.forward()` 的 TND 数据流与 kernel 参数；
- 认出训练 recipe 打开 varlen 的开关：`model_spec`、`override.imports` 与 CP 版 `parallelize_fn`；
- 复用仓库单测覆盖的 CP 元数据边界 `QwenCPMetadata`。


<figure>
  <img src="images/06.05_varlen_training_path.png" alt="Varlen Attention 训练接入路径：DataLoader 用 greedy packing 让每条样本的 positions 从 0 重新编号；trainer 调用 get_attention_masks，Decoder 按 inner attention 类型生成 VarlenMetadata；parallelize_qwen3_5_cp 写入 context_parallel_mesh 并把元数据整理成 QwenCPMetadata；AscVarlenAttention 把 BSND 转成 TND，用 actual_seq_qlen 与 sparse_mode=7 调用 CANN npu_fusion_attention_v3。" loading="lazy" style="max-width: 100%; height: auto;">
  <figcaption>图 06.05-1：<code>positions</code> 归零处产生的样本边界，经 <code>VarlenMetadata</code>、<code>QwenCPMetadata</code> 传到 CANN FA v3 的 TND 分段计算；带 ①②③ 的节点由 <code>qwen35_27b_long_text_sft()</code> 的三个开关打开。</figcaption>
</figure>


## 1. 后端选择：`attn_backend` 决定 inner attention

`model_registry(flavor, attn_backend="flex")` 把 backend 名字写进模型配置，最终由 `torchtitan/models/common/config_utils.py` 的 `get_attention_config()` 映射成具体的 inner attention Config：

| `attn_backend` | inner attention | 掩码来源 |
|---|---|---|
| `flex` | `FlexAttention.Config` | `positions` → document-aware `BlockMask` |
| `flex_flash` | `FlexAttention.Config(block_size=(256, 128), kernel_options={"BACKEND": "FLASH"})` | 同上，仅 Hopper/Blackwell |
| `varlen` | `VarlenAttention.Config` | `positions` → `VarlenMetadata(cu_seq_q, cu_seq_k, max_q, max_k)` |
| `sdpa` | —— | 语言模型不再接受，直接 `ValueError` |

`get_attention_config()` 的 docstring 写明了原因：语言模型的数据加载器始终输出逐文档 `positions`，因此每个 backend 都必须是能消费文档边界的 masked attention；`ScaledDotProductAttention` 只能接收一个 `is_causal` 布尔量，所以不再是语言模型 backend。


In [ ]:
import torch
import torchtitan_npu  # noqa: F401  # 载入 NPU override 与 patch

from torchtitan.models.common.attention import VarlenAttention
from torchtitan.models.common.config_utils import get_attention_config

for backend in ('flex', 'varlen'):
    print(f'{backend:8s} -> {type(get_attention_config(backend)).__qualname__}')

try:
    get_attention_config('sdpa')
except ValueError as error:
    print('sdpa     -> ValueError:', error)

assert type(get_attention_config('varlen')) is VarlenAttention.Config


## 2. 样本边界：`positions` 归零处就是文档起点

上游 `ChatDataset` 用 greedy packing 填满一个定长容器：每条样本的 token 顺序不变，进入新样本时 `positions` 重新从 0 开始编号；末尾补 padding 时，padding 也从 0 开始，于是单独构成最后一个区间。

训练器每步在 `torchtitan/trainer.py` 里做两件事：把 `positions` 传给模型，并在 inner attention 属于 `FlexAttention.Config` 或 `VarlenAttention.Config` 时调用 `model.get_attention_masks(positions=...)`，把结果作为 `attention_masks` 一起送入 forward。`Decoder.get_attention_masks()` 随后按类型分派：

- `FlexAttention.Config` → `BlockMask`；
- `VarlenAttention.Config` → `create_varlen_metadata_for_document(positions)`。

`create_varlen_metadata_for_document()` 的实现可以概括成三步：找出 `positions == 0` 的位置；拼上容器长度 `seq_len` 得到每个 batch 的区间端点；按 batch 累加偏移后补上总长度，装进 `VarlenMetadata(cu_seq_q, cu_seq_k, max_q, max_k, cu_seq_q_host)`。`cu_seq_q` 位于设备上且为 `int32`，`cu_seq_q_host` 只在显式要求时生成。


In [ ]:
from torchtitan.models.common.attention import (
    VarlenMetadata,
    create_varlen_metadata_for_document,
)

# 一个容器内的三条样本（3 + 4 + 1 token）与 2 个 padding token，positions 每次归零
positions = torch.tensor([[0, 1, 2, 0, 1, 2, 3, 0, 0]])

metadata = create_varlen_metadata_for_document(positions)
print('cu_seq_q    :', metadata.cu_seq_q.tolist(), metadata.cu_seq_q.dtype)
print('max_q       :', metadata.max_q)
host = create_varlen_metadata_for_document(positions, include_host_offsets=True)
print('host offsets:', host.cu_seq_q_host)

assert isinstance(metadata, VarlenMetadata)
assert metadata.cu_seq_q.tolist() == [0, 3, 7, 8, 9]  # 3 条样本 + 1 个 padding 区间
assert metadata.cu_seq_q.tolist() == metadata.cu_seq_k.tolist()
assert metadata.max_q == 4


### 2.1 override 生效之后，mask dispatch 仍然命中 varlen 分支

`AscVarlenAttention.Config` 继承上游 `VarlenAttention.Config`，所以换成 NPU 实现之后，训练器与 `Decoder` 里那两处 `isinstance(..., VarlenAttention.Config)` 判断继续成立，仍然产出 `VarlenMetadata`：

```text
override.imports 里列出 asc_cp
  → apply_overrides() 把 model_spec.model.layers.*.attention.inner_attention
     从 VarlenAttention.Config 换成 AscVarlenAttention.Config
  → trainer 与 Decoder 的 varlen 判断照旧命中
  → attention_masks 仍是 VarlenMetadata，只是计算改由 CANN 融合算子完成
```


## 3. CP 边界：`VarlenMetadata` → `QwenCPMetadata`

Qwen3.5/3.6 的 CP 路线由 `torchtitan_npu/override/qwen3_5/parallelize.py` 的 `parallelize_qwen3_5_cp()` 接管并行化：

```text
parallelize_qwen3_5_cp(model, parallel_dims=...)
  ├─ mesh = parallel_dims.get_mesh("cp")
  ├─ model.context_parallel_mesh = mesh
  ├─ model.register_forward_pre_hook(prepare_sequence_metadata, with_kwargs=True)
  └─ 每个 block 的 inner attention / delta net 也写入 context_parallel_mesh
```

`prepare_sequence_metadata()` 在模型 forward 之前把传入的 varlen 元数据交给 `build_sequence_metadata()`，后者产出 `QwenCPMetadata`：

| 字段 | 内容 |
|---|---|
| `cu_seqlens` | `cu_seq_q` 转成 `int64` |
| `cu_seqlens_cpu` | 设备张量的 CPU 副本，供 kernel 侧读取分段长度 |
| `actual_seq_qlen` | `cu_seqlens_cpu[1:]`，即每个分段的累计结束位置 |

CP 下若 KV 需要跨 rank gather（元数据带 `k_global_gather_indices`），`build_sequence_metadata()` 会先用 `cu_seq_q` 与 `cu_seq_k` 的差分还原真实文档起点，在 CP mesh 上做一次 `Shard(1) → Replicate` 的 redistribute，再重算 `cu_seqlens`。仓库用 `tests/unit_tests/override/qwen3_5/test_parallelize.py` 把这个 CPU 边界固定下来，下面的代码直接复用同一条路径。


In [ ]:
from types import SimpleNamespace

from torchtitan_npu.override.qwen3_5.parallelize import (
    QwenCPMetadata,
    build_sequence_metadata,
)

varlen = SimpleNamespace(cu_seq_q=torch.tensor([0, 3, 8], dtype=torch.int32))
cp_metadata = build_sequence_metadata(varlen, None, batch=1, length=8)

print('type           :', type(cp_metadata).__qualname__)
print('cu_seqlens     :', cp_metadata.cu_seqlens.tolist(),
      cp_metadata.cu_seqlens.dtype, cp_metadata.cu_seqlens.device)
print('actual_seq_qlen:', cp_metadata.actual_seq_qlen.tolist())

assert isinstance(cp_metadata, QwenCPMetadata)
assert cp_metadata.cu_seqlens.dtype == torch.int64
assert cp_metadata.cu_seqlens.device.type == 'cpu'
assert cp_metadata.actual_seq_qlen.tolist() == [3, 8]


## 4. 内核接入：`AscVarlenAttention.forward()`

本章 Wordle TND 路线用的实现是 `torchtitan_npu/override/qwen3/varlen_attention.py`（随 03.03 的 patch 一起提供）。它继承上游 `VarlenAttention`，只替换计算部分，forward 的每一步都有明确职责：

1. **取分段长度**：`actual_seq = attention_masks.cu_seq_q.to(torch.int64).cpu()[1:]` —— FA v3 要求 CPU 上的累计结束位置，去掉首个 0；
2. **转 TND**：`[B, L, N, D]` 展平成 `[B * L, N, D]`，并统一为 `bfloat16`；
3. **掩码与 kernel**：`torch.ops.npu.npu_fusion_attention_v3()`，`input_layout="TND"`、`sparse_mode=7`、`pre_tockens=length`、`next_tockens=0`，`actual_seq_qlen` 与 `actual_seq_kvlen` 取同一份分段长度；`atten_mask` 是缓存的 2048×2048 上三角 causal 掩码，按设备复用；
4. **还原布局**：kernel 输出 view 回 `[B, L, N, D]`，再转回输入 dtype。

GQA 直接由张量形状表达（Qwen3-1.7B 是 16 个 query 头、8 个 KV 头），不需要显式 repeat；样本隔离由第 3 步的两个 `actual_seq_*` 参数表达——分段内做因果计算，分段之间不产生可见 pair。

仓库里带 Context Parallel 的同类实现是 `torchtitan_npu/override/qwen3_5/varlen_attention.py`（`asc_cp`），它在同一位置多出 §3 的 `context_parallel_mesh` 头/序列交换；本节这条 qwen3 路线是非 CP 版本。


In [ ]:
import os
from pathlib import Path

ttnpu_dir = Path(os.environ.get('TTNPU_DIR', '/mnt/workspace/gitCode/cann/torchtitan-npu-wordle-latest'))
source = (ttnpu_dir / 'torchtitan_npu/override/qwen3/varlen_attention.py').read_text()


def window(text: str, start_marker: str, end_marker: str) -> str:
    start = text.index(start_marker)
    return text[start:text.index(end_marker, start)].rstrip()


for argument in (
    'input_layout="TND"',
    'sparse_mode=_SPARSE_MODE_VARLEN',
    '_SPARSE_MODE_VARLEN = 7',
    'pre_tockens=length',
    'next_tockens=0',
    'actual_seq_qlen=actual_seq',
    'actual_seq_kvlen=actual_seq',
    'attention_masks.cu_seq_q.to(torch.int64).cpu()[1:]',
):
    print(f'{argument:52s} in source: {argument in source}')

print('=' * 60)
print(window(source, '        output = torch.ops.npu.npu_fusion_attention_v3(', '@override('))


### 4.1 override 工厂：`derive()` 复制共享字段

`asc` 是 `--override.imports` 的入口。装饰器声明 target 与 FQN 匹配范围，工厂函数用 `derive()` 把共享字段从上游 Config 复制到 `AscVarlenAttention.Config`，只声明真正变化的差异，因此 `window_size` 这类上游字段自动保留；`exact=True` 表示只替换类型恰好是 `VarlenAttention.Config` 的节点。

```python
@override(
    target=VarlenAttention.Config,
    fqns=["model_spec.model.layers.*.attention.inner_attention"],
    exact=True,
    description="Use CANN TND attention for Qwen3 varlen layers",
)
def asc(cfg: VarlenAttention.Config) -> AscVarlenAttention.Config:
    return derive(cfg, AscVarlenAttention.Config)
```


In [ ]:
from torchtitan_npu.override.qwen3.varlen_attention import (
    AscVarlenAttention,
    asc,
)

upstream = VarlenAttention.Config(window_size=(512, 0))
npu_config = asc(upstream)

print('upstream:', type(upstream).__qualname__, upstream.window_size)
print('npu     :', type(npu_config).__qualname__, npu_config.window_size)

assert isinstance(npu_config, AscVarlenAttention.Config)
assert isinstance(npu_config, VarlenAttention.Config)
assert npu_config.window_size == (512, 0)


## 5. 训练任务怎样打开这条路线

`torchtitan_npu/models/qwen3/config_registry.py` 里的 `sft_qwen3_1_7b_wordle_tnd()` 把同一份 Wordle SFT 配置切到 TND：数据、optimizer、checkpoint 全部继承 `sft_qwen3_1_7b_wordle()`，只改两个开关。

```python
def sft_qwen3_1_7b_wordle_tnd() -> Trainer.Config:
    config = sft_qwen3_1_7b_wordle()
    config.model_spec = model_registry("1.7B", attn_backend="varlen")   # ① 选上游 varlen contract
    config.override.imports.append(
        "torchtitan_npu.override.qwen3.varlen_attention.asc"            # ② 换成 AscVarlenAttention
    )
    return config
```

- ① 让 `Decoder.get_attention_masks()` 走 varlen 分支，产出 `VarlenMetadata`；
- ② 让 override 系统在 build 之前把 `VarlenAttention.Config` 换成 `AscVarlenAttention.Config`。

override 的生效时机在上游 `torchtitan/trainer.py`：`model_config.update_from_config(...)` 之后、`model_config.build()` 之前调用 `apply_overrides(config.override, config)`。

启动沿用仓内 launcher，config registry 通过 `--module` 指定：

```bash
NGPU=2 \
MODULE=torchtitan_npu.models.qwen3 \
CONFIG=sft_qwen3_1_7b_wordle_tnd \
./scripts/run_train.sh --training.steps 1 --training.seq-len 4096
```

**一个容易踩的坑**：`--override.imports` 一旦在命令行给出，就会替换 recipe 里的整个列表，而不是追加。所以叠加其它 override（例如 profiler 的 `cann`）时必须把 varlen 这条一起写上：

```bash
--override.imports torchtitan_npu.override.qwen3.varlen_attention.asc \
  'torchtitan_npu.override.common.profiler.cann={"profile_ranks":[0],"profile_with_memory":true}'
```


In [ ]:
recipe_source = (ttnpu_dir / 'torchtitan_npu/models/qwen3/config_registry.py').read_text()

for line in (
    'def sft_qwen3_1_7b_wordle_tnd() -> Trainer.Config:',
    'config = sft_qwen3_1_7b_wordle()',
    'model_registry("1.7B", attn_backend="varlen")',
    'torchtitan_npu.override.qwen3.varlen_attention.asc',
):
    print(f'{line:58s} in source: {line in recipe_source}')

print('=' * 60)
print(recipe_source[recipe_source.index('def sft_qwen3_1_7b_wordle_tnd()'):].rstrip())


## 6. 计算区域：分段因果的 pair 数

Varlen 的价值来自把文档边界显式交给 kernel。长度为 `L` 的文档在因果 Attention 下有 `L(L+1)/2` 个可见 query-key pair，各文档 pair 之和就是融合算子需要覆盖的计算区域。下面用 06.02 的 Wordle 长度分布算一遍这个工作量模型：non-greedy 的四条样本各占一个 1024 容器，greedy packing 把它们放进同一个容器，varlen 则按文档长度分段。


In [ ]:
def causal_pairs(length: int) -> int:
    return length * (length + 1) // 2


seq_len = 1024
document_lengths = [180, 260, 410, 120]
padding = seq_len - sum(document_lengths)

non_greedy_dense = len(document_lengths) * causal_pairs(seq_len)
packed_dense = causal_pairs(seq_len)
segmented = sum(causal_pairs(length) for length in document_lengths) + causal_pairs(padding)

print(f'non-greedy dense candidate pairs: {non_greedy_dense:,}')
print(f'one packed dense container:       {packed_dense:,}')
print(f'segmented varlen pairs:           {segmented:,}')
print(f'dense-packed / segmented ratio:   {packed_dense / segmented:.2f}x')

assert segmented < packed_dense < non_greedy_dense


上面的 ratio 是工作量模型：真实速度还受 tiling、head shape、backward、metadata 构造、D2H、layout、其他模型层与通信影响。端到端收益以训练日志的 step time 与 trace 为准。


## 7. 验证清单

1. `config.model_spec.model.layers` 上逐层确认 `inner_attention` 已是 `AscVarlenAttention.Config`；
2. `create_varlen_metadata_for_document(positions)` 的分段数与 DataLoader 的真实样本数加 padding 区间一致；
3. 同一样本内部的多轮消息保持可见，不同样本之间的 attention probability 为 0；
4. 有效 token 的输出与 Q/K/V 梯度在既定 BF16 容差内一致；
5. 同一 checkpoint 与 batch 下的 loss、grad norm 可解释；
6. trace 中 Attention 命中 `npu_fusion_attention_v3` 对应的融合算子。


## 练习

1. （单选题）当前版本里，让语言模型 Attention 消费文档边界的两种 backend 是？
    A. `sdpa` 与 `flex`
    B. `flex` 与 `varlen`
    C. `sdpa` 与 `varlen`
    D. 只有 `sdpa`

2. （判断题）`create_varlen_metadata_for_document()` 以 `positions == 0` 的位置作为文档起点，因此容器末尾的 padding 也会形成一个独立区间。

3. （单选题）`AscVarlenAttention.forward()` 把文档边界交给 CANN FA v3 的参数组合是？
    A. `input_layout="BSND"` + `sparse_mode=0`
    B. `input_layout="TND"` + `actual_seq_qlen` / `actual_seq_kvlen` + `sparse_mode=7`
    C. 只设置 `is_causal=True`
    D. 只把 padding 的 label 改成 `-100`

4. （多选题）把 recipe 切到 NPU varlen 路线，需要同时满足哪些条件？
    A. `model_spec` 选择 `attn_backend="varlen"`
    B. `override.imports` 里包含 `torchtitan_npu.override.qwen3.varlen_attention.asc`
    C. 命令行给出 `--override.imports` 时，把这条 varlen override 一并列出
    D. 把 `window_size` 手工改成 `(-1, -1)`


In [ ]:
!cat ./answer/06.05_answer.txt
